In [1]:
import os
import re
import pandas as pd
import pickle
import numpy as np
from nltk import word_tokenize
from nltk.tag import RegexpTagger, UnigramTagger, BigramTagger, TrigramTagger
from natasha import (
    Segmenter,
    MorphVocab,
    NewsEmbedding,
    NewsMorphTagger,
    NewsSyntaxParser,
    NewsNERTagger,
    PER, LOC, ORG,
    NamesExtractor,
    DatesExtractor,
    MoneyExtractor,
    AddrExtractor,
    Doc,    
)
from sklearn_crfsuite import CRF, metrics
from sklearn.model_selection import train_test_split, cross_val_score
from typing import List, Dict, Any
from collections import defaultdict, Counter

from navec import Navec
from slovnet import NER, Syntax
from ipymarkup import show_span_ascii_markup as show_markup
from razdel import sentenize, tokenize
from corus import load_ne5

import nltk
from nltk.corpus import stopwords
from string import punctuation
import matplotlib.pyplot as plt

In [2]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download('maxent_ne_chunker')
nltk.download('words')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


True

In [3]:
# !wget https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar
# !wget https://storage.yandexcloud.net/natasha-slovnet/packs/slovnet_ner_news_v1.tar

In [4]:
COL_NAME = "text"
DATA_PATH = "./les/_data"
DATA_COL_PATH = f"{DATA_PATH}/Collection5"

In [5]:
# Инициализация необходимых компонентов Natasha
segmenter = Segmenter()
morph_vocab = MorphVocab()

emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
syntax_parser = NewsSyntaxParser(emb)
ner_tagger = NewsNERTagger(emb)

names_extractor = NamesExtractor(morph_vocab)
dates_extractor = DatesExtractor(morph_vocab)
money_extractor = MoneyExtractor(morph_vocab)
addr_extractor = AddrExtractor(morph_vocab)

In [63]:
# Функция загрузки и обработки данных
def load_and_process_data(directory: str) -> pd.DataFrame:
    """
    Загружает текстовые файлы и соответствующие им аннотации из директории.
    Выполняет проверку на соответствие аннотаций тексту.

    Args:
        directory: Путь к директории с файлами.

    Returns:
        DataFrame с текстами и аннотациями.
    """
    files: List[Dict[str, Any]] = []
    error_count = 0

    for filename in pd.Series(
            [e.split(".")[0] for e in os.listdir(directory)]
    ).unique():
        data: Dict[str, Any] = {}
        try:
            # Чтение текстового файла с обработкой декодирования
            with open(
                    os.path.join(directory, filename + ".txt"),
                    encoding="utf-8",
                    errors="ignore",
            ) as f:
                text = f.read().replace("\n", "  ")

            # Чтение разметки, если она присутствует
            ann_file = os.path.join(directory, filename + ".ann")
            if os.path.exists(ann_file):
                with open(ann_file, 'r', encoding='utf-8') as f:
                    lines = f.readlines()

                entities = []
                for line in lines:
                    parts = line.strip().split('\t')
                    if len(parts) == 3:
                        num, entity_info, text_span = parts
                        entity_type, start, end = entity_info.split(' ')
                        start, end = int(start), int(end)
                        if text[start:end] != text_span:
                            raise ValueError("Text mismatch between annotation and actual text")
                        entities.append({
                            'label': entity_type,
                            'start': start,
                            'end': end
                        })
            else:
                entities = []

            # Сохранение текста и списка сущностей
            data["text"] = text
            data["entities"] = entities

            files.append(data)

        except Exception as e:
            print(f"Error processing file {filename}: {e}")
            error_count += 1

    print(f"Processing completed with {error_count} errors.")
    return pd.DataFrame(files)

In [7]:
# Функция для разметки на основе правил (rule-based labeling)
def apply_rule_based_labeling(data: pd.DataFrame) -> pd.DataFrame:
    """
    Применяет разметку на основе правил к DataFrame с текстами.

    Args:
        data: DataFrame с текстами.

    Returns:
        DataFrame с добавленным столбцом 'entities_new' с разметкой.
    """
    # Регулярные выражения для разметки имен собственных, дат и числовых значений
    patterns = [
        (
            r"\b[А-ЯЁ][а-яё]+(?:\s[А-ЯЁ][а-яё]+){1,2}\b",
            "PER",
        ),
        (r"\b[A-Z][a-z]+(?:\s[A-Z][a-z]+){1,2}\b", "PER"),
        (
            r"\b\d{1,2}\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+\d{4}\b",
            "DATE",
        ),
        (r"\b\d{1,2}\.\d{1,2}\.\d{4}\b", "DATE"),
        (r"\b\d{4}\b", "DATE"),  # Год
        (
            r"\b\d+\s*(тысяч|тыс|млн|млрд|трлн)?\b",
            "NUM",
        ), 
        (r"\b\d+\.?\d*\b", "NUM"),
        (
            r"\b(?:г\.\s*)?[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?\b",
            "LOC",
        ),
        (
            r"\b[А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+)*\s+(ООО|ЗАО|ОАО|ПАО|АО)\b",
            "ORG",
        ), 
        (r"\b(?:Inc|Corp|Ltd|LLC|Co)\b", "ORG"),
    ]

    regexp_tagger = RegexpTagger(patterns)

    # Добавление нового столбца в DataFrame
    data["entities_new"] = None

    # Обход каждой строки в DataFrame
    for idx, row in data.iterrows():
        text = row[COL_NAME]
        tokens = word_tokenize(text)
        tagged_tokens = regexp_tagger.tag(tokens)

        # Использование дополнительных тэггеров для улучшения разметки
        unigram_tagger = UnigramTagger([tagged_tokens], backoff=regexp_tagger)
        bigram_tagger = BigramTagger([tagged_tokens], backoff=unigram_tagger)
        trigram_tagger = TrigramTagger([tagged_tokens], backoff=bigram_tagger)

        rule_based_tags = trigram_tagger.tag(tokens)

        # Сохранение разметки в новый столбец
        entities = []
        for i, (word, tag) in enumerate(rule_based_tags):
            if tag != "O" and tag is not None:
                start_pos = text.find(word)
                end_pos = start_pos + len(word)
                entities.append({"label": tag, "start": start_pos, "end": end_pos})
        data.at[idx, "entities_new"] = entities

    return data

In [39]:
# Функция для разметки с помощью Natasha
def apply_natasha_labeling(data: pd.DataFrame) -> pd.DataFrame:
    """
    Применяет разметку Natasha к DataFrame с текстами.

    Args:
        data: DataFrame с текстами.

    Returns:
        DataFrame с добавленным столбцом 'entities_new' с разметкой Natasha.

    """
    # Добавление нового столбца в DataFrame
    data["entities_new"] = None

    # Обход каждой строки в DataFrame
    for idx, row in data.iterrows():        
        text = row[COL_NAME]
        doc = Doc(text)
        doc.segment(segmenter)
        doc.tag_ner(ner_tagger)

        # for token in doc.tokens:
        #     token.lemmatize(morph_vocab)

        doc.parse_syntax(syntax_parser)
        entities = []
        # Сохранение разметки в новый столбец
        for span in doc.spans:
            span.normalize(morph_vocab)
            entities.append({"label": span.type, "start": span.start, "end": span.stop})

        data.at[idx, "entities_new"] = entities

    return data

In [9]:
# Функция для подготовки данных для CRF
def prepare_data_for_crf(data: pd.DataFrame, use_entities_new: bool = True):
    """
    Преобразует данные в формат, подходящий для обучения CRF.

    Args:
        data: DataFrame с текстами и аннотациями.
        use_entities_new: Использовать ли столбец 'entities_new' для аннотаций.

    Returns:
        Список списков токенов и список списков меток.
    """
    X, y = [], []
    for _, row in data.iterrows():
        tokens = word_tokenize(row[COL_NAME])
        labels = ["O"] * len(tokens)

        # Если используется 'entities_new', и если она присутствует
        if use_entities_new and "entities_new" in row and row["entities_new"]:
            for entity in row["entities_new"]:
                entity_text = row[COL_NAME][entity["start"] : entity["end"]]
                entity_tokens = word_tokenize(entity_text)

                try:
                    start_idx = tokens.index(entity_tokens[0])
                    end_idx = start_idx + len(entity_tokens)
                    labels[start_idx:end_idx] = [entity["label"]] * (
                        end_idx - start_idx
                    )
                except ValueError:
                    print(f"Entity '{entity_text}' not found in tokens.")

        X.append(tokens)
        y.append(labels)

    return X, y

In [10]:
# Функция для подготовки данных для CRF
def prepare_tokenized_data_for_crf(data: pd.DataFrame):
    """
    Преобразует данные, подготовленные с помощью get_tokenized_df, в формат, подходящий для обучения CRF.

    Args:
        data: DataFrame с токенами и метками (выход get_tokenized_df).

    Returns:
        Список списков токенов и список списков меток.
    """
    X, y = [], []
    grouped = data.groupby(
        data.index
    )  # Группируем по индексу для восстановления исходных текстов

    for _, group in grouped:
        tokens = group["token"].tolist()  # Извлекаем токены
        labels = group["label"].tolist()  # Извлекаем соответствующие метки

        X.append(tokens)
        y.append(labels)

    return X, y

In [ ]:
# Функции для обработки данных

def clean_text(text):
    # Замена несущественных спец. символов (кроме точек и дефисов) на пробелы
    text = re.sub(r"[^\w\s.,-]", " ", text)
    return text


def preprocess_text(text):
    text = clean_text(text)
    return text


def preprocess_dataframe(df, text_column):
    df[COL_NAME] = df[text_column].apply(
        lambda x: preprocess_text(x)
    )
    return df

In [11]:
def save_to_pickle(df, filename):
    with open(filename, "wb") as f:
        pickle.dump(df, f)

In [71]:
def get_tokenized(data: pd.DataFrame, entities_col_name="entities_new") -> pd.DataFrame:
    df = data.explode(entities_col_name)
    df = pd.concat([df, df[entities_col_name].apply(pd.Series)], axis=1)

    # Разбиение на токены
    df["tokens"] = df[COL_NAME].apply(word_tokenize)

    # Создание нового DataFrame
    new_df = []
    for _, row in df.iterrows():
        for i, token in enumerate(row["tokens"]):
            start = row[COL_NAME].find(token)
            end = start + len(token)
            new_df.append(
                {
                    "token": token,
                    "label": (
                        row["label"]
                        if start == row["start"] and end == row["end"]
                        else "O"
                    ),
                    "start": start,
                    "end": end,
                }
            )

    return pd.DataFrame(new_df)

## Основная часть скрипта

In [80]:
test_article = ""

with open(f'{DATA_COL_PATH}/147.txt', encoding="utf-8") as file:
    test_article = [line for line in file]
test_article = '\n'.join(test_article)
test_article = test_article.replace('\n','')
test_article[:300]

'Посла США в Исландии вызвали для объяснений по делу WikileaksПосла США в Рейкьявике Луиса Арреагу (Luis Arreaga) вызвали в министерство иностранных Исландии после того, как американский суд обязал Twitter раскрыть персональные данные депутата исландского парламента Биргиты Йонсдотир. Об этом сообщае'

In [13]:
navec = Navec.load(f'{DATA_PATH}/navec_news_v1_1B_250K_300d_100q.tar')
ner = NER.load(f'{DATA_PATH}/slovnet_ner_news_v1.tar')
ner.navec(navec)
markup = ner(test_article)
show_markup(markup.text, markup.spans)

Посла США в Исландии вызвали для объяснений по делу WikileaksПосла США
      LOC   LOC─────                                ORG──────      LOC
 в Рейкьявике Луиса Арреагу (Luis Arreaga) вызвали в министерство 
   LOC─────── PER─────────────────────────                        
иностранных Исландии после того, как американский суд обязал Twitter 
            LOC─────                                         ORG──── 
раскрыть персональные данные депутата исландского парламента Биргиты 
                                                             PER─────
Йонсдотир. Об этом сообщает Associated Press со ссылкой на 
─────────                   ORG─────────────               
представителя МИДа. Йонсдотир однажды сотрудничала с WikiLeaks.Когда 
              ORG─  PER──────                        ORG──────       
именно состоялась встреча, неясно. В посольстве США агентству 
                                                LOC           
отказались комментировать ситуацию до понедельника, 10 янв

In [14]:
doc_test_article = Doc(test_article)
doc_test_article.segment(segmenter)
doc_test_article.tag_ner(ner_tagger)

doc_test_article.parse_syntax(syntax_parser)
for token in doc_test_article.spans:
    print(token.text, token.type, token.start, token.stop)

США LOC 6 9
Исландии LOC 12 20
Wikileaks ORG 52 61
США LOC 67 70
Рейкьявике LOC 73 83
Луиса Арреагу (Luis Arreaga) PER 84 112
Исландии LOC 148 156
Twitter ORG 197 204
Биргиты Йонсдотир PER 266 283
Associated Press ORG 302 318
МИДа ORG 347 351
Йонсдотир PER 353 362
WikiLeaks ORG 386 395
США LOC 450 453
Twitter ORG 548 555
WikiLeaks ORG 635 644
Исландии LOC 673 681
Соединенные Штаты LOC 777 794
Исландии LOC 845 853
Огмундур Йохансон (Ogmundur Jonasson) PER 935 972
Йонсдотир PER 983 992
США LOC 1093 1096
WikiLeaks ORG 1140 1149
Джулиан Ассанж PER 1150 1164
Великобритании LOC 1193 1207
WikiLeaks ORG 1234 1243
Ираке LOC 1330 1335
Афганистане LOC 1338 1349
Йонсдотир PER 1394 1403
Ираке LOC 1480 1485
Reuters ORG 1550 1557
США LOC 1565 1568
WikiLeaks ORG 1594 1603
Ассанжа PER 1643 1650


In [64]:
# Загрузка и обработка данных
df = load_and_process_data(DATA_COL_PATH)
df

Error processing file last_37: Text mismatch between annotation and actual text
Error processing file last_60: Text mismatch between annotation and actual text
Processing completed with 2 errors.


,text,entities
0,Россия рассчитывает на конструктивное воздейст...,"[{'label': 'GEOPOLIT', 'start': 0, 'end': 6}, ..."
1,Комиссар СЕ критикует ограничительную политику...,"[{'label': 'ORG', 'start': 9, 'end': 11}, {'la..."
2,"Пулеметы, автоматы и снайперские винтовки изъя...","[{'label': 'LOC', 'start': 82, 'end': 89}, {'l..."
3,4 октября назначены очередные выборы Верховног...,"[{'label': 'ORG', 'start': 37, 'end': 54}, {'l..."
4,Следственное управление при прокуратуре требуе...,"[{'label': 'ORG', 'start': 0, 'end': 39}, {'la..."
...,...,...
993,"Депутат от ""ЕР"": К отставке А.Сердюкова причас...","[{'label': 'ORG', 'start': 11, 'end': 15}, {'l..."
994,Си Цзиньпин избран генсеком Коммунистической...,"[{'label': 'PER', 'start': 2, 'end': 13}, {'la..."
995,"""Ведомости"" узнали о смене лидера московских е...","[{'label': 'MEDIA', 'start': 0, 'end': 11}, {'..."
996,СМИ узнали о кутежах туркменского чиновника на...,"[{'label': 'MEDIA', 'start': 0, 'end': 3}, {'l..."


In [65]:
copy_df = df.copy()

In [66]:
text_lengths = df["text"].apply(len)
print(f"Среднее количество слов статьи - {np.average(text_lengths):.0f} слов.")

Среднее количество слов статьи - 1708 слов.


In [67]:
# Предобработка датафрейма
df = preprocess_dataframe(df, "text")

In [68]:
df

,text,entities
0,Россия рассчитывает на конструктивное воздейст...,"[{'label': 'GEOPOLIT', 'start': 0, 'end': 6}, ..."
1,Комиссар СЕ критикует ограничительную политику...,"[{'label': 'ORG', 'start': 9, 'end': 11}, {'la..."
2,"Пулеметы, автоматы и снайперские винтовки изъя...","[{'label': 'LOC', 'start': 82, 'end': 89}, {'l..."
3,4 октября назначены очередные выборы Верховног...,"[{'label': 'ORG', 'start': 37, 'end': 54}, {'l..."
4,Следственное управление при прокуратуре требуе...,"[{'label': 'ORG', 'start': 0, 'end': 39}, {'la..."
...,...,...
993,Депутат от ЕР К отставке А.Сердюкова причас...,"[{'label': 'ORG', 'start': 11, 'end': 15}, {'l..."
994,Си Цзиньпин избран генсеком Коммунистической...,"[{'label': 'PER', 'start': 2, 'end': 13}, {'la..."
995,Ведомости узнали о смене лидера московских е...,"[{'label': 'MEDIA', 'start': 0, 'end': 11}, {'..."
996,СМИ узнали о кутежах туркменского чиновника на...,"[{'label': 'MEDIA', 'start': 0, 'end': 3}, {'l..."


In [69]:
# Разделение данных на обучающую и тестовую выборки
train, test = train_test_split(df, test_size=0.2, random_state=42)

# Разделение обучающей выборки для Natasha и Rule-based labeling
natasha_data, rule_based_data = train_test_split(train, test_size=0.15, random_state=42)

# Применение разметки
rule_based_labeled_data = apply_rule_based_labeling(rule_based_data.copy())
natasha_labeled_data = apply_natasha_labeling(natasha_data.copy())
# 
# Объединение данных
combined_data = pd.concat([natasha_labeled_data, rule_based_labeled_data])
save_to_pickle(combined_data, "combined_data.pkl")
combined_data

,text,entities,entities_new
179,Обама представил кандидатуру нового министра ф...,"[{'label': 'PER', 'start': 0, 'end': 5}, {'lab...","[{'label': 'LOC', 'start': 65, 'end': 68}, {'l..."
175,Глава московского СКП А.Багмет окончательно ли...,"[{'label': 'ORG', 'start': 18, 'end': 21}, {'l...","[{'label': 'ORG', 'start': 18, 'end': 21}, {'l..."
212,Похороны советской науки реформа РАН застала ...,"[{'label': 'ORG', 'start': 34, 'end': 37}, {'l...","[{'label': 'ORG', 'start': 34, 'end': 37}, {'l..."
80,Ю.Тимошенко не будет оспаривать результаты пер...,"[{'label': 'PER', 'start': 0, 'end': 11}, {'la...","[{'label': 'LOC', 'start': 75, 'end': 82}, {'l..."
631,А.Бабаков назвал неизбежным свой выход из Спр...,"[{'label': 'PER', 'start': 0, 'end': 9}, {'lab...","[{'label': 'PER', 'start': 0, 'end': 9}, {'lab..."
...,...,...,...
529,Дмитрий Пестов назначен вице-премьером подмо...,"[{'label': 'PER', 'start': 2, 'end': 16}, {'la...","[{'label': 'LOC', 'start': 2, 'end': 9}, {'lab..."
279,СК обнаружил крупные махинации на Балтий...,"[{'label': 'ORG', 'start': 6, 'end': 8}, {'lab...","[{'label': 'LOC', 'start': 40, 'end': 50}, {'l..."
769,Д.Медведев уволил ряд высокопоставленных военн...,"[{'label': 'PER', 'start': 0, 'end': 10}, {'la...","[{'label': 'LOC', 'start': 52, 'end': 61}, {'l..."
918,В ЦИК Белоруссии не связывают убийство члена к...,"[{'label': 'ORG', 'start': 2, 'end': 5}, {'lab...","[{'label': 'LOC', 'start': 6, 'end': 16}, {'la..."


In [73]:
tokenized_data_old = get_tokenized(combined_data, entities_col_name="entities")
tokenized_data_old

,token,label,start,end
0,Обама,PER,0,5
1,представил,O,6,16
2,кандидатуру,O,17,28
3,нового,O,29,35
4,министра,O,36,44
...,...,...,...,...
7256252,таможенных,O,1950,1960
7256253,и,O,1,2
7256254,налоговых,O,1963,1972
7256255,сборов,O,1973,1979


Количество классов дефолтного датасета

In [74]:
tokenized_data_old.label.value_counts()

label
O           7245138
GEOPOLIT       3131
ORG            3095
PER            2673
LOC            1577
MEDIA           643
Name: count, dtype: int64

In [75]:
tokenized_data = get_tokenized(combined_data)
tokenized_data

,token,label,start,end
0,Обама,O,0,5
1,представил,O,6,16
2,кандидатуру,O,17,28
3,нового,O,29,35
4,министра,O,36,44
...,...,...,...,...
7842747,таможенных,O,1950,1960
7842748,и,O,1,2
7842749,налоговых,O,1963,1972
7842750,сборов,O,1973,1979


Заметно, что в новой разметке регулярками и Наташей добавилось LOC меток и не стало меток GEOPOLIT. И присутствует дисбаланс классов, класс O представляет подавляющее большинство данных. Интересно, что читая этот же датасет из библиотеки corus, value_counts тоже будут отличаться.

In [77]:
tokenized_data.label.value_counts()

label
O       7823777
LOC       12858
ORG        3360
PER        2069
NUM         446
DATE        242
Name: count, dtype: int64

In [82]:
tokenized_test = get_tokenized(test, entities_col_name="entities")
tokenized_test

,token,label,start,end
0,Министр,O,0,7
1,топлива,O,8,15
2,и,O,1,2
3,энергетики,O,18,28
4,Норвегии,GEOPOLIT,29,37
...,...,...,...,...
1330031,ситуации,O,4505,4513
1330032,в,O,3,4
1330033,Республике,O,4516,4526
1330034,Беларусь,O,4031,4039


In [83]:
tokenized_test.label.value_counts()

label
O           1327496
GEOPOLIT        724
ORG             706
PER             603
LOC             339
MEDIA           168
Name: count, dtype: int64

In [85]:
# Подготовка данных для CRF
X_train, y_train = prepare_tokenized_data_for_crf(tokenized_data)
X_test, y_test = prepare_tokenized_data_for_crf(tokenized_test)

In [93]:
# Обучение модели CRF
crf = CRF(algorithm="lbfgs", max_iterations=1000)

In [87]:
# Кросс-валидация
scores = cross_val_score(crf, X_train, y_train, cv=5)  # 5-fold cross-validation
print("Cross-validation scores:", scores)
print("Average F1-score:", scores.mean())

Cross-validation scores: [0.9982844  0.99850435 0.99844825 0.99857894 0.99371585]
Average F1-score: 0.9975063597859481


In [88]:
# Обучение на всех обучающих данных
crf.fit(X_train, y_train)

CRF(algorithm='lbfgs', max_iterations=100)

In [96]:
len(X_train), len(y_train)

(7842752, 7842752)

In [89]:
# Оценка модели
y_pred = crf.predict(X_test)
print(metrics.flat_classification_report(y_test, y_pred, labels=crf.classes_))

Q:\code\envs\_win10\py312_math\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
Q:\code\envs\_win10\py312_math\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
Q:\code\envs\_win10\py312_math\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
Q:\code\envs\_win10\py3

              precision    recall  f1-score   support

           O       1.00      1.00      1.00   1327496
         LOC       0.00      0.00      0.00       339
         ORG       0.02      0.00      0.00       706
         PER       0.00      0.00      0.00       603
         NUM       0.00      0.00      0.00         0
        DATE       0.00      0.00      0.00         0

   micro avg       1.00      1.00      1.00   1329144
   macro avg       0.17      0.17      0.17   1329144
weighted avg       1.00      1.00      1.00   1329144



Q:\code\envs\_win10\py312_math\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
Q:\code\envs\_win10\py312_math\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
Q:\code\envs\_win10\py312_math\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [97]:
# Пример использования обученной модели
new_text = "Сегодня, 10 октября 2023 года, компания ООО 'Рога и копыта' заключила контракт на поставку 10 тысяч единиц продукции."
new_tokens = word_tokenize(new_text)
new_labels = crf.predict([new_tokens])
print("Разметка нового текста:", new_labels)

AttributeError: 'NoneType' object has no attribute 'tag'